In [ ]:
import tensorflow as tf

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.20.0


In [6]:
!ls

archive.zip  sample_data


In [7]:
import zipfile

zip_ref = zipfile.ZipFile("archive.zip", "r")
zip_ref.extractall("dataset")
zip_ref.close()

print("Dataset Extracted Successfully")

Dataset Extracted Successfully


In [8]:
!ls dataset

plantvillage  PlantVillage


In [9]:
!ls dataset/PlantVillage

Pepper__bell___Bacterial_spot  Tomato_Late_blight
Pepper__bell___healthy	       Tomato_Leaf_Mold
Potato___Early_blight	       Tomato_Septoria_leaf_spot
Potato___healthy	       Tomato_Spider_mites_Two_spotted_spider_mite
Potato___Late_blight	       Tomato__Target_Spot
Tomato_Bacterial_spot	       Tomato__Tomato_mosaic_virus
Tomato_Early_blight	       Tomato__Tomato_YellowLeaf__Curl_Virus
Tomato_healthy


In [10]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_gen.flow_from_directory(
    "dataset/PlantVillage",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    subset="training"
)

val_data = train_gen.flow_from_directory(
    "dataset/PlantVillage",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    subset="validation"
)

Found 16516 images belonging to 15 classes.
Found 4122 images belonging to 15 classes.


In [11]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)

x = Dense(
    128,
    activation="relu"
)(x)

predictions = Dense(
    train_data.num_classes,
    activation="softmax"
)(x)

model = Model(
    inputs=base_model.input,
    outputs=predictions
)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,423,887 (9.25 MB)

 Trainable params: 165,903 (648.06 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [12]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
517/517 ━━━━━━━━━━━━━━━━━━━━ 92s 137ms/step - accuracy: 0.8231 - loss: 0.5650 - val_accuracy: 0.8797 - val_loss: 0.3497
Epoch 2/5
517/517 ━━━━━━━━━━━━━━━━━━━━ 36s 69ms/step - accuracy: 0.9123 - loss: 0.2647 - val_accuracy: 0.9073 - val_loss: 0.2801
Epoch 3/5
517/517 ━━━━━━━━━━━━━━━━━━━━ 36s 70ms/step - accuracy: 0.9300 - loss: 0.2017 - val_accuracy: 0.9064 - val_loss: 0.2699
Epoch 4/5
517/517 ━━━━━━━━━━━━━━━━━━━━ 38s 74ms/step - accuracy: 0.9467 - loss: 0.1601 - val_accuracy: 0.9051 - val_loss: 0.2781
Epoch 5/5
517/517 ━━━━━━━━━━━━━━━━━━━━ 36s 70ms/step - accuracy: 0.9626 - loss: 0.1200 - val_accuracy: 0.9253 - val_loss: 0.2361


In [13]:
model.save("sakthi_crop_model.h5")

print("Model Saved")

Model Saved


In [15]:
from tensorflow.keras.preprocessing import image
import numpy as np

img = image.load_img(
    "test_leaf.png",
    target_size=(224,224)
)

img_array = image.img_to_array(img)

img_array = np.expand_dims(
    img_array,
    axis=0
)

img_array = img_array / 255.0

prediction = model.predict(img_array)

predicted_class = np.argmax(prediction)

print("Class Index:", predicted_class)
print("Confidence:", np.max(prediction))

1/1 ━━━━━━━━━━━━━━━━━━━━ 13s 13s/step
Class Index: 3
Confidence: 0.99927527


In [16]:
class_names = list(train_data.class_indices.keys())

predicted_class = np.argmax(prediction)

print("Disease:", class_names[predicted_class])

print(
    "Confidence:",
    round(float(np.max(prediction))*100,2),
    "%"
)

Disease: Potato___Late_blight
Confidence: 99.93 %


In [17]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

print("Conversion Successful")

Saved artifact at '/tmp/tmpwbwow299'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 15), dtype=tf.float32, name=None)
Captures:
  140527899470800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140527899473104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140527899475792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140527899475408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140527899474256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140527899475600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140527899474064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140527899475216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140527899475024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140527899474448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1405278994746

In [18]:
with open(
    "sakthi_crop_model.tflite",
    "wb"
) as f:

    f.write(tflite_model)

print("TFLite Model Saved")

TFLite Model Saved


In [19]:
!ls


archive.zip  sakthi_crop_model.h5      sample_data
dataset      sakthi_crop_model.tflite  test_leaf.png
